Importaciones

In [1]:
import pandas as pd
import scipy.stats as stats
from scipy.stats import spearmanr
import numpy as np

In [2]:
df = pd.read_csv('../output/df_final.csv')

### Temperatura media vs. ventas

In [3]:
temperatura_fecha = (
    df.assign(fecha=pd.to_datetime(df["fecha"]))
      .groupby("fecha")["media"]
      .mean()
      .rename("temperatura_media")
)

cantidad_fecha = (
    df.assign(fecha=pd.to_datetime(df["fecha"]))
      .groupby("fecha")["cantidad"]
      .sum()
)

ventas_temperatura = (
    cantidad_fecha.rename("cantidad_ventas")
    .rename_axis("fecha")
    .reset_index()
)

ventas_temperatura["fecha"] = pd.to_datetime(ventas_temperatura["fecha"])

ventas_temperatura = ventas_temperatura.merge(
    temperatura_fecha.reset_index(),
    on="fecha",
    how="inner"
)

ventas_temperatura

,fecha,cantidad_ventas,temperatura_media
0,2021-01-02,54039.5677,26.9
1,2021-01-03,52387.8631,28.3
2,2021-01-04,60503.8856,27.0
3,2021-01-05,52937.7395,25.5
4,2021-01-06,42781.1186,26.4
...,...,...,...
266,2021-09-26,50281.8482,23.1
267,2021-09-27,44041.0166,27.2
268,2021-09-28,36640.0411,23.8
269,2021-09-29,31324.0311,24.1


Verificación de supuestos

In [4]:
e, p = stats.shapiro(ventas_temperatura["cantidad_ventas"])
print("cantidad_ventas:")
print("Estadistico:", e, "p-valor:", p)
e, p = stats.shapiro(ventas_temperatura["temperatura_media"])
print("temperatura_media:")
print("Estadistico:", e, "p-valor:", p)

cantidad_ventas:
Estadistico: 0.9260386554645217 p-valor: 2.3248652064357254e-10
temperatura_media:
Estadistico: 0.9534918818854761 p-valor: 1.3243466379649028e-07


Prueba rho de Spearman

In [5]:
rho, p = spearmanr(
    ventas_temperatura["cantidad_ventas"],
    ventas_temperatura["temperatura_media"]
)

print("Coeficiente rho:", rho)
print("p-valor:", p)

Coeficiente rho: 0.14241800543841338
p-valor: 0.01899505472840049


### Precipitaciones vs. ventas

In [6]:
precipitaciones_fecha = (
    df.assign(fecha=pd.to_datetime(df["fecha"]))
      .groupby("fecha")["precipitacion"]
      .mean()
      .rename("precipitaciones")
)

cantidad_fecha = (
    df.assign(fecha=pd.to_datetime(df["fecha"]))
      .groupby("fecha")["cantidad"]
      .sum()
)

ventas_precipitaciones = (
    cantidad_fecha.rename("cantidad_ventas")
    .rename_axis("fecha")
    .reset_index()
)

ventas_precipitaciones["fecha"] = pd.to_datetime(ventas_precipitaciones["fecha"])

ventas_precipitaciones = ventas_precipitaciones.merge(
    precipitaciones_fecha.reset_index(),
    on="fecha",
    how="inner"
)

ventas_precipitaciones

,fecha,cantidad_ventas,precipitaciones
0,2021-01-02,54039.5677,0.0
1,2021-01-03,52387.8631,0.0
2,2021-01-04,60503.8856,8.6
3,2021-01-05,52937.7395,0.0
4,2021-01-06,42781.1186,7.4
...,...,...,...
266,2021-09-26,50281.8482,0.0
267,2021-09-27,44041.0166,0.0
268,2021-09-28,36640.0411,0.8
269,2021-09-29,31324.0311,0.0


Verificación de supuestos

In [7]:
e, p = stats.shapiro(ventas_precipitaciones["cantidad_ventas"])
print("cantidad_ventas:")
print("Estadistico:", e, "p-valor:", p)
e, p = stats.shapiro(ventas_precipitaciones["precipitaciones"])
print("precipitaciones:")
print("Estadistico:", e, "p-valor:", p)

cantidad_ventas:
Estadistico: 0.9260386554645217 p-valor: 2.3248652064357254e-10
precipitaciones:
Estadistico: 0.3135353710759028 p-valor: 1.4021880007583188e-30


Prueba rho de Spearman

In [8]:
rho, p = spearmanr(
    ventas_precipitaciones["cantidad_ventas"],
    ventas_precipitaciones["precipitaciones"]
)

print("Coeficiente rho:", rho)
print("p-valor:", p)

Coeficiente rho: 0.03349278077946027
p-valor: 0.5830286355728919


### Temperatura media vs. secciones

In [9]:
categorias = pd.cut(df['media'], bins=[-float('inf'), 10, 20, 30, float('inf')], labels=['< 10°', '10° - 20°', '20° - 30°', '> 30°'])

secciones_temperatura = df.cantidad.groupby([categorias, df['desc_secciones']]).sum().unstack(fill_value=0).T
secciones_temperatura

media,< 10°,10° - 20°,20° - 30°,> 30°
desc_secciones,,,,
Almacen,67323.7708,8.061248e+05,1.858010e+06,22209.7244
Bazar,19395.4700,1.962227e+05,4.623398e+05,7654.3600
Bebidas,10854.0000,1.580341e+05,4.501851e+05,5447.0000
Belleza y Salud,8588.0000,1.095850e+05,2.737360e+05,3179.0000
Hogar,506.0000,1.970000e+03,2.759000e+03,33.0000
Insumos,14238.0000,1.528180e+05,1.946120e+05,7954.0000
Limpieza,11898.8450,1.517053e+05,4.044051e+05,4542.0950
Perecederos,99572.5701,1.400621e+06,3.618245e+06,43254.9412
Perecederos Terceros,0.0000,3.550000e-01,9.400000e-01,0.0000


Verificación de supuestos

In [10]:
# 1. Ejecutar Chi2 y extraer frecuencias esperadas
chi2_stat, p_val, dof, esperados = stats.chi2_contingency(secciones_temperatura)

# 2. VERIFICACIÓN DEL SUPUESTO DE FRECUENCIAS ESPERADAS
num_total_casillas = esperados.size
casillas_menor_5 = np.sum(esperados < 5)
porcentaje_menor_5 = (casillas_menor_5 / num_total_casillas) * 100
casillas_menor_1 = np.sum(esperados < 1)

print("=== VERIFICACIÓN DE SUPUESTOS ===")
print(f"Total de casillas: {num_total_casillas}")
print(
    f"Casillas con E < 5: {casillas_menor_5} ({porcentaje_menor_5:.1f}% del"
    " total)"
)
print(f"Casillas con E < 1: {casillas_menor_1}")

# Diagnóstico de validez de la prueba
if porcentaje_menor_5 <= 20 and casillas_menor_1 == 0:
    print(
        "-> CUMPLE SUPUESTOS: La prueba de Chi-cuadrado es estadísticamente"
        " válida."
    )
else:
    print(
        "-> ADVERTENCIA: NO cumple supuestos. Se recomienda agrupar categorías"
        " o usar simulaciones de Monte Carlo."
    )

print(f"\np-valor de la prueba: {p_val:.4e}")

=== VERIFICACIÓN DE SUPUESTOS ===
Total de casillas: 44
Casillas con E < 5: 4 (9.1% del total)
Casillas con E < 1: 4
-> ADVERTENCIA: NO cumple supuestos. Se recomienda agrupar categorías o usar simulaciones de Monte Carlo.

p-valor de la prueba: 0.0000e+00


No se cumplen supuestos, se procede a filtrar categorías irrelevantes para el análisis

In [11]:
# Identificamos el volumen de cantidad total por sección para ver cuáles son irrelevantes o muy pequeñas
resumen_secciones = df.groupby('desc_secciones')['cantidad'].sum().sort_values()
print("Volumen de ventas por sección:")
print(resumen_secciones)

# Filtramos las secciones irrelevantes (por ejemplo, 'Perecederos Terceros' y otras sumamente bajas si las hubiera)
secciones_a_excluir = ['Perecederos Terceros']
df_filtrado = df[~df['desc_secciones'].isin(secciones_a_excluir)].copy()

# Volvemos a categorizar la temperatura con el DataFrame filtrado
categorias_filtradas = pd.cut(df_filtrado['media'], bins=[-float('inf'), 10, 20, 30, float('inf')], labels=['< 10°', '10° - 20°', '20° - 30°', '> 30°'])

# Creamos la nueva tabla de contingencia
secciones_temp_filtrada = df_filtrado.cantidad.groupby([categorias_filtradas, df_filtrado['desc_secciones']]).sum().unstack(fill_value=0).T
print("\nNueva Tabla de Contingencia (Filtrada):")
display(secciones_temp_filtrada)

# Ejecutamos nuevamente la prueba de Chi-cuadrado y validamos supuestos
chi2_stat, p_val, dof, esperados = stats.chi2_contingency(secciones_temp_filtrada)

num_total_casillas = esperados.size
casillas_menor_5 = np.sum(esperados < 5)
porcentaje_menor_5 = (casillas_menor_5 / num_total_casillas) * 100
casillas_menor_1 = np.sum(esperados < 1)

print("=== NUEVA VERIFICACIÓN DE SUPUESTOS ===")
print(f"Total de casillas: {num_total_casillas}")
print(f"Casillas con E < 5: {casillas_menor_5} ({porcentaje_menor_5:.1f}% del total)")
print(f"Casillas con E < 1: {casillas_menor_1}")

if porcentaje_menor_5 <= 20 and casillas_menor_1 == 0:
    print("-> CUMPLE SUPUESTOS: La prueba de Chi-cuadrado ahora es estadísticamente válida.")
else:
    print("-> ADVERTENCIA: Aún no cumple supuestos. Se sugiere agrupar más categorías o usar simulación.")

print(f"\np-valor de la prueba: {p_val:.4e}")
print("chi2_stat:", chi2_stat)

Volumen de ventas por sección:
desc_secciones
Perecederos Terceros    1.295000e+00
Hogar                   5.268000e+03
Tienda                  1.619200e+05
Insumos                 3.696220e+05
Belleza y Salud         3.950880e+05
Limpieza                5.725513e+05
Bebidas                 6.245202e+05
Bazar                   6.856123e+05
Produccion              1.532547e+06
Almacen                 2.753668e+06
Perecederos             5.161694e+06
Name: cantidad, dtype: float64

Nueva Tabla de Contingencia (Filtrada):


media,< 10°,10° - 20°,20° - 30°,> 30°
desc_secciones,,,,
Almacen,67323.7708,8.061248e+05,1.858010e+06,22209.7244
Bazar,19395.4700,1.962227e+05,4.623398e+05,7654.3600
Bebidas,10854.0000,1.580341e+05,4.501851e+05,5447.0000
Belleza y Salud,8588.0000,1.095850e+05,2.737360e+05,3179.0000
Hogar,506.0000,1.970000e+03,2.759000e+03,33.0000
Insumos,14238.0000,1.528180e+05,1.946120e+05,7954.0000
Limpieza,11898.8450,1.517053e+05,4.044051e+05,4542.0950
Perecederos,99572.5701,1.400621e+06,3.618245e+06,43254.9412
Produccion,47232.6925,4.283556e+05,1.043179e+06,13779.8482


=== NUEVA VERIFICACIÓN DE SUPUESTOS ===
Total de casillas: 40
Casillas con E < 5: 0 (0.0% del total)
Casillas con E < 1: 0
-> CUMPLE SUPUESTOS: La prueba de Chi-cuadrado ahora es estadísticamente válida.

p-valor de la prueba: 0.0000e+00
chi2_stat: 69217.93452073802


Ahora se cumplen supuestos, se calcula V de Crámer

In [12]:
# El tamaño total de la muestra (N) es la suma de todas las frecuencias observadas
n = secciones_temp_filtrada.sum().sum()

# Número de filas (r) y columnas (c) de la tabla de contingencia
r, c = secciones_temp_filtrada.shape

# El mínimo entre (r-1) y (c-1)
min_dimension = min(r - 1, c - 1)

# Fórmula de la V de Crámer
cramers_v = np.sqrt(chi2_stat / (n * min_dimension))

print("=== COEFICIENTE V DE CRÁMER ===")
print(f"V de Crámer: {cramers_v:.4f}")

# Interpretación estándar
if cramers_v < 0.10:
    print("Interpretación: Asociación muy débil o insignificante.")
elif cramers_v < 0.30:
    print("Interpretación: Asociación débil.")
elif cramers_v < 0.50:
    print("Interpretación: Asociación moderada.")
else:
    print("Interpretación: Asociación fuerte.")

=== COEFICIENTE V DE CRÁMER ===
V de Crámer: 0.0434
Interpretación: Asociación muy débil o insignificante.


### Precipitaciones vs. secciones

In [13]:
categorias = pd.cut(df['precipitacion'], bins=[-0.01, 0, 10, 25, 50, float('inf')], labels=['Sin lluvia', 'Ligera', 'Moderada', 'Fuerte', 'Muy fuerte'])

secciones_precipitacion = df.cantidad.groupby([categorias, df['desc_secciones']]).sum().unstack(fill_value=0).T
secciones_precipitacion

precipitacion,Sin lluvia,Ligera,Moderada,Fuerte,Muy fuerte
desc_secciones,,,,,
Almacen,2.086269e+06,414336.6594,144905.0325,67032.9965,41124.3336
Bazar,5.202239e+05,103883.7454,35941.0176,14892.5872,10671.0935
Bebidas,4.712522e+05,85939.6400,42541.0250,15036.0000,9751.2850
Belleza y Salud,2.980180e+05,59293.0000,22256.0000,9051.0000,6470.0000
Hogar,3.996000e+03,870.0000,242.0000,103.0000,57.0000
Insumos,3.049091e+05,44227.2500,13610.6750,5055.0000,1820.0000
Limpieza,4.272195e+05,87637.2821,34276.8880,13750.1810,9667.4352
Perecederos,3.844863e+06,824529.2390,284416.1404,128751.0770,79134.4510
Perecederos Terceros,1.115000e+00,0.1800,0.0000,0.0000,0.0000


Verificacion de supuestos

In [14]:
# 1. Ejecutar Chi2 y extraer frecuencias esperadas
chi2_stat, p_val, dof, esperados = stats.chi2_contingency(secciones_precipitacion)

# 2. VERIFICACIÓN DEL SUPUESTO DE FRECUENCIAS ESPERADAS
num_total_casillas = esperados.size
casillas_menor_5 = np.sum(esperados < 5)
porcentaje_menor_5 = (casillas_menor_5 / num_total_casillas) * 100
casillas_menor_1 = np.sum(esperados < 1)

print("=== VERIFICACIÓN DE SUPUESTOS ===")
print(f"Total de casillas: {num_total_casillas}")
print(
    f"Casillas con E < 5: {casillas_menor_5} ({porcentaje_menor_5:.1f}% del"
    " total)"
)
print(f"Casillas con E < 1: {casillas_menor_1}")

# Diagnóstico de validez de la prueba
if porcentaje_menor_5 <= 20 and casillas_menor_1 == 0:
    print(
        "-> CUMPLE SUPUESTOS: La prueba de Chi-cuadrado es estadísticamente"
        " válida."
    )
else:
    print(
        "-> ADVERTENCIA: NO cumple supuestos. Se recomienda agrupar categorías"
        " o usar simulaciones de Monte Carlo."
    )

print(f"\np-valor de la prueba: {p_val:.4e}")

=== VERIFICACIÓN DE SUPUESTOS ===
Total de casillas: 55
Casillas con E < 5: 5 (9.1% del total)
Casillas con E < 1: 5
-> ADVERTENCIA: NO cumple supuestos. Se recomienda agrupar categorías o usar simulaciones de Monte Carlo.

p-valor de la prueba: 0.0000e+00


No se cumplen supuestos, se procede a filtrar categorías irrelevantes para el análisis

In [15]:
# Filtramos la categoría irrelevante 'Perecederos Terceros' para el análisis de precipitación
secciones_a_excluir = ['Perecederos Terceros']
df_filtrado_precip = df[~df['desc_secciones'].isin(secciones_a_excluir)].copy()

# Volvemos a categorizar la precipitación con el DataFrame filtrado
categorias_precip_filtradas = pd.cut(df_filtrado_precip['precipitacion'], bins=[-0.01, 0, 10, 25, 50, float('inf')], labels=['Sin lluvia', 'Ligera', 'Moderada', 'Fuerte', 'Muy fuerte'])

# Creamos la nueva tabla de contingencia filtrada
secciones_precip_filtrada = df_filtrado_precip.cantidad.groupby([categorias_precip_filtradas, df_filtrado_precip['desc_secciones']]).sum().unstack(fill_value=0).T
print("Nueva Tabla de Contingencia de Precipitación (Filtrada):")
display(secciones_precip_filtrada)

# Ejecutamos nuevamente la prueba de Chi-cuadrado y validamos supuestos
chi2_stat_p, p_val_p, dof_p, esperados_p = stats.chi2_contingency(secciones_precip_filtrada)

num_total_casillas_p = esperados_p.size
casillas_menor_5_p = np.sum(esperados_p < 5)
porcentaje_menor_5_p = (casillas_menor_5_p / num_total_casillas_p) * 100
casillas_menor_1_p = np.sum(esperados_p < 1)

print("=== NUEVA VERIFICACIÓN DE SUPUESTOS (PRECIPITACIÓN) ===")
print(f"Total de casillas: {num_total_casillas_p}")
print(f"Casillas con E < 5: {casillas_menor_5_p} ({porcentaje_menor_5_p:.1f}% del total)")
print(f"Casillas con E < 1: {casillas_menor_1_p}")

if porcentaje_menor_5_p <= 20 and casillas_menor_1_p == 0:
    print("-> CUMPLE SUPUESTOS: La prueba de Chi-cuadrado para precipitación ahora es estadísticamente válida.")
else:
    print("-> ADVERTENCIA: Aún no cumple supuestos. Se sugiere agrupar más categorías.")

print(f"\np-valor de la prueba: {p_val_p:.4e}")
print("chi2_stat:", chi2_stat)

Nueva Tabla de Contingencia de Precipitación (Filtrada):


precipitacion,Sin lluvia,Ligera,Moderada,Fuerte,Muy fuerte
desc_secciones,,,,,
Almacen,2.086269e+06,414336.6594,144905.0325,67032.9965,41124.3336
Bazar,5.202239e+05,103883.7454,35941.0176,14892.5872,10671.0935
Bebidas,4.712522e+05,85939.6400,42541.0250,15036.0000,9751.2850
Belleza y Salud,2.980180e+05,59293.0000,22256.0000,9051.0000,6470.0000
Hogar,3.996000e+03,870.0000,242.0000,103.0000,57.0000
Insumos,3.049091e+05,44227.2500,13610.6750,5055.0000,1820.0000
Limpieza,4.272195e+05,87637.2821,34276.8880,13750.1810,9667.4352
Perecederos,3.844863e+06,824529.2390,284416.1404,128751.0770,79134.4510
Produccion,1.128643e+06,258232.2685,71577.8568,39708.1057,34385.7496


=== NUEVA VERIFICACIÓN DE SUPUESTOS (PRECIPITACIÓN) ===
Total de casillas: 50
Casillas con E < 5: 0 (0.0% del total)
Casillas con E < 1: 0
-> CUMPLE SUPUESTOS: La prueba de Chi-cuadrado para precipitación ahora es estadísticamente válida.

p-valor de la prueba: 0.0000e+00
chi2_stat: 27191.728529988482


Ahora sí se cumplen los supuestos. Se calcula la V de Crámer

In [16]:
# Tamaño total de la muestra (N) para precipitaciones
n_p = secciones_precip_filtrada.sum().sum()

# Dimensiones de la tabla de contingencia (filas, columnas)
r_p, c_p = secciones_precip_filtrada.shape

# Mínimo entre (filas-1) y (columnas-1)
min_dimension_p = min(r_p - 1, c_p - 1)

# Fórmula de la V de Crámer
cramers_v_p = np.sqrt(chi2_stat_p / (n_p * min_dimension_p))

print("=== COEFICIENTE V DE CRÁMER (PRECIPITACIÓN) ===")
print(f"V de Crámer: {cramers_v_p:.4f}")

# Interpretación estándar
if cramers_v_p < 0.10:
    print("Interpretación: Asociación muy débil o insignificante.")
elif cramers_v_p < 0.30:
    print("Interpretación: Asociación débil.")
elif cramers_v_p < 0.50:
    print("Interpretación: Asociación moderada.")
else:
    print("Interpretación: Asociación fuerte.")

=== COEFICIENTE V DE CRÁMER (PRECIPITACIÓN) ===
V de Crámer: 0.0235
Interpretación: Asociación muy débil o insignificante.
